In [3]:
# CELL 1 — Environment Setup (Updated for Colab 2025)
import os

# Fix: update package list first, then install with fallback flag
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless --fix-missing -qq

# Upgrade to PySpark 4.0 to match Colab's environment
!pip install pyspark==4.0.0 --quiet

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# Verify Java is actually there
java_check = os.popen("java -version 2>&1").read()
print(f"☕ Java: {java_check}")
print("✅ Environment ready.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../openjdk-11-jre-headless_11.0.31+11-1ubuntu1~22.04.2_amd64.deb ...
Unpacking openjdk-11-jre-headless:amd64 (11.0.31+11-1ubuntu1~22.04.2) ...
Selecting previously unselected package openjdk-11-jdk-headless:amd64.
Preparing to unpack .../openjdk-11-jdk-headless_11.0.31+11-1ubuntu1~22.04.2_amd64.deb ...
Unpacking openjdk-11-jdk-headless:amd64 (11.0.31+11-1ubuntu1~22.04.2) ...
Setting up openjdk-11-jre-headless:amd64 (11.0.31+11-1ubuntu1~22.04.2) ...
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/jjs to provide /usr/bin/jjs (jjs) in auto mode
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/rmid to provid

In [4]:
# CELL 2 — SparkSession
from pyspark.sql import SparkSession
import os

# Updated for Java 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

spark = SparkSession.builder \
    .appName("LocalPulseAI") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.host", "localhost") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark version: {spark.version}")
print("✅ SparkSession is live — LocalPulse AI engine started.")

✅ Spark version: 4.0.0
✅ SparkSession is live — LocalPulse AI engine started.


In [6]:
# CELL 3 — Mount Drive + Load Cleaned Data
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

df_tfidf = spark.read.parquet("/content/drive/MyDrive/LocalPulseAI/processed/tfidf_features/")

print(f"✅ Loaded! Rows: {df_tfidf.count():,}")
df_tfidf.printSchema()

Mounted at /content/drive
✅ Loaded! Rows: 6,989,408
root
 |-- review_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- label: double (nullable = true)
 |-- features: vector (nullable = true)
 |-- text: string (nullable = true)



In [7]:
#Cell 4 - Loading the rows

df_tfidf = spark.read.parquet("/content/drive/MyDrive/LocalPulseAI/processed/tfidf_features")

print(f"✅ Loaded! Rows: {df_tfidf.count():,}")
df_tfidf.printSchema()

✅ Loaded! Rows: 6,989,408
root
 |-- review_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- label: double (nullable = true)
 |-- features: vector (nullable = true)
 |-- text: string (nullable = true)



In [8]:
#Cell 5 - Splitting the dataset into training data (0.8) and testing data (0.2)

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

train_data, test_data = df_tfidf.randomSplit([0.8, 0.2], seed=42)

print(f"✅ Split complete!")
print(f"🏋️ Training rows : {train_data.count():,}")
print(f"🧪 Testing rows  : {test_data.count():,}")

✅ Split complete!
🏋️ Training rows : 5,590,035
🧪 Testing rows  : 1,399,373


In [9]:
#Cell 6 - Train the Classifier

lr = LogisticRegression(featuresCol = "features", labelCol = "label", maxIter = 20)
print("Training the model...(this will take a few minutes be patient!)")

lr_model = lr.fit(train_data)
print("✅ Model trained!")

Training the model...(this will take a few minutes be patient!)
✅ Model trained!


In [10]:
# CELL 7 — Evaluate the Model

predictions = lr_model.transform(test_data)
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print(f"✅ Model Accuracy: {accuracy * 100:.2f}%")

# Breakdown — see how it performed on each sentiment class
predictions.groupBy("sentiment", "prediction").count().orderBy("sentiment").show()

✅ Model Accuracy: 86.38%
+---------+----------+------+
|sentiment|prediction| count|
+---------+----------+------+
| negative|       0.0| 36682|
| negative|       1.0|272392|
| negative|       2.0| 13736|
|  neutral|       0.0| 69077|
|  neutral|       2.0| 32980|
|  neutral|       1.0| 36617|
| positive|       0.0|903439|
| positive|       1.0| 18589|
| positive|       2.0| 15861|
+---------+----------+------+



In [14]:
# CELL 7 — Smart Alert System
from pyspark.sql.functions import udf, col, when
from pyspark.sql.types import StringType

CRITICAL_KEYWORDS = [
    "cockroach", "roach", "rat ", "rats ", "mice", "mouse",
    "maggot", "food poisoning", "food poison", "got sick",
    "felt sick after", "vomit", "diarrhea", "mold on",
    "mould on", "raw chicken", "undercooked chicken",
    "hair in my", "bug in my", "worm in", "maggots",
    "dead rat", "dead mouse", "pest", "infestation"
]

WARNING_KEYWORDS = [
    "very rude", "extremely rude", "so rude",
    "cold food", "wrong order", "wrong item",
    "waited over an hour", "ignored us",
    "stale bread", "expired", "smelled bad",
    "smelled off", "terrible smell"
]

def get_alert_level(text):
    if text is None:
        return "normal"
    text_lower = text.lower()
    for keyword in CRITICAL_KEYWORDS:
        if keyword in text_lower:
            return "CRITICAL"
    for keyword in WARNING_KEYWORDS:
        if keyword in text_lower:
            return "WARNING"
    return "normal"

alert_udf = udf(get_alert_level, StringType())

df_alerts = predictions.withColumn("alert_level", alert_udf(col("text")))

df_smart_alerts = df_alerts.withColumn(
    "alert_level",
    when(
        (col("alert_level") == "CRITICAL") & (col("sentiment") == "negative"),
        "CRITICAL"
    ).when(
        (col("alert_level") == "WARNING") & (col("sentiment") == "negative"),
        "WARNING"
    ).otherwise("normal")
)

print("🚨 SMART CRITICAL ALERTS:")
print("=" * 80)
df_smart_alerts.filter(col("alert_level") == "CRITICAL") \
    .select("business_id", "stars", "sentiment", "alert_level", "text") \
    .show(20, truncate=100)

print(f"\n📊 CRITICAL : {df_smart_alerts.filter(col('alert_level') == 'CRITICAL').count():,}")
print(f"⚠️  WARNING  : {df_smart_alerts.filter(col('alert_level') == 'WARNING').count():,}")
print(f"✅ Normal   : {df_smart_alerts.filter(col('alert_level') == 'normal').count():,}")

🚨 SMART CRITICAL ALERTS:
+----------------------+-----+---------+-----------+----------------------------------------------------------------------------------------------------+
|           business_id|stars|sentiment|alert_level|                                                                                                text|
+----------------------+-----+---------+-----------+----------------------------------------------------------------------------------------------------+
|EhypF8VPpIbLvt0JLx0Qew|  1.0| negative|   CRITICAL|Large hair in my taco and I paid for a large fry and it wasn't in my bag. Pretty disappointed thi...|
|pEpb16y8Wfe7EmJgusu6gw|  2.0| negative|   CRITICAL|We camped here in December 2016 in our motorhome. The camp host was pretty rude. We came in from ...|
|KZNgXQJ9Qq2LNi_xOebSvg|  1.0| negative|   CRITICAL|I came to puccinos to grab a quick coffee before class one evening. As I was going back to my car...|
|UQMUtySn2q9cSObiw5UVsw|  2.0| negative|   CRITICAL

In [15]:
# CELL 9 — Business Owner Dashboard
from pyspark.sql.functions import count, round, avg

df_summary = df_alerts.groupBy("business_id").agg(
    count("*").alias("total_reviews"),
    round(avg("stars"), 2).alias("avg_stars"),
    count(when(col("sentiment") == "positive", 1)).alias("positive_count"),
    count(when(col("sentiment") == "negative", 1)).alias("negative_count"),
    count(when(col("sentiment") == "neutral", 1)).alias("neutral_count"),
    count(when(col("alert_level") == "CRITICAL", 1)).alias("critical_alerts"),
    count(when(col("alert_level") == "WARNING", 1)).alias("warning_alerts")
)

print("📊 Business Owner Dashboard — Top 10 by critical alerts:")
df_summary.orderBy(col("critical_alerts").desc()).show(10, truncate=20)

📊 Business Owner Dashboard — Top 10 by critical alerts:
+--------------------+-------------+---------+--------------+--------------+-------------+---------------+--------------+
|         business_id|total_reviews|avg_stars|positive_count|negative_count|neutral_count|critical_alerts|warning_alerts|
+--------------------+-------------+---------+--------------+--------------+-------------+---------------+--------------+
|YiSr-W5BBX6uXnhWP...|           82|     3.85|            59|            16|            7|             41|             0|
|ozOneB4jXOD6hv5WB...|          114|     4.73|           109|             4|            1|             34|             0|
|8kUh6TROemLfbVR_e...|          147|     3.89|           112|            15|           20|             31|             0|
|6ajnOk0GcY9xbb5Oc...|          596|     4.25|           490|            36|           70|             30|             5|
|ueAkLzWFFTzQkq3jz...|          163|     3.99|           127|            16|           20|

In [17]:
# CELL 10 — Save Model + Results
lr_model.write().overwrite().save("/content/drive/MyDrive/LocalPulseAI/models/lr_sentiment_model")

df_smart_alerts.select("review_id", "business_id", "stars", "sentiment", "alert_level", "text") \
    .write.parquet("/content/drive/MyDrive/LocalPulseAI/outputs/review_alerts", mode="overwrite")

df_summary.write.parquet("/content/drive/MyDrive/LocalPulseAI/outputs/business_summary", mode="overwrite")

print("✅ Model saved!     → LocalPulseAI/models/lr_sentiment_model")
print("✅ Alerts saved!    → LocalPulseAI/outputs/review_alerts")
print("✅ Dashboard saved! → LocalPulseAI/outputs/business_summary")
print("🎉 ML work complete!")

✅ Model saved!     → LocalPulseAI/models/lr_sentiment_model
✅ Alerts saved!    → LocalPulseAI/outputs/review_alerts
✅ Dashboard saved! → LocalPulseAI/outputs/business_summary
🎉 ML work complete!
